# 🏗️ Week 3: The RAG "ETL" Pipeline
**From Data Engineer to AI Architect**

## 🎯 Objective
In this notebook, we stop treating RAG (Retrieval Augmented Generation) as "magic" and start treating it as a **Data Engineering Pipeline**.

We will build a "Glass Box" system where you can see exactly what happens at every stage:
1.  **[E]xtract:** **Loading** a raw PDF (The Course Curriculum).
2.  **[T]ransform:** **Chunking** the text into semantic windows.
3.  **[L]oad:** **Embedding** and indexing the vectors into `ChromaDB`.

## 🛠️ Setup & Prerequisites
1.  **API Keys:** Ensure you have a `.env` file in this directory with your `OPENAI_API_KEY`.
2.  **The Data:** This notebook is pre-configured to use `AI Architect Study Guide - 8 Week Curriculum.pdf` (The 8-Week Roadmap in the `curriculum` directory).
3.  **Install Dependencies:**
    \`\`\`bash
    pip install langchain langchain-openai langchain-community chromadb pymupdf jupyter
    \`\`\`

## 🧪 How to Test This System
Once you run all the cells, you will enter an interactive **QA Loop**. Use these specific test cases to verify the architecture:

### 1. The "Happy Path" (Verifying Retrieval)
Ask specific questions about the hardcoded curriculum to prove the RAG is working.
* *Prompt:* "What are we learning in Week 4?"
* *Expected:* It should detail "Advanced RAG," "Re-ranking," or "Vector Databases" (based on the PDF content).
* *Why:* GPT-4o alone doesn't know the specific schedule of *this* private course.

### 2. The "Synthesis" Test (Verifying Context Window)
Ask a question that requires combining information from multiple chunks.
* *Prompt:* "Compare the topics of Week 2 and Week 6."
* *Expected:* It should pull chunks from both weeks and summarize the difference.

### 3. The "Guardrail" Test (Verifying Grounding)
Ask a question completely outside the scope of the document.
* *Prompt:* "How do I deploy a Kubernetes cluster?" or "Who won the 2024 Super Bowl?"
* *Expected:* **"I don't know based on the provided document."**
* *Why:* We set `temperature=0` and used a strict system prompt to prevent hallucination.

## ⚠️ Outliers & Gotchas (What to Watch For)
* **The "Lost in the Middle" Phenomenon:** If you ask a very broad question (e.g., "Summarize the whole course"), the system might miss details in the middle weeks because we are only retrieving `k=3` chunks.
* **Chunk Boundaries:** If a sentence is cut in half between Chunk A and Chunk B, the semantic meaning might be lost. (This is why we use `chunk_overlap`).

In [13]:
import os
import sys
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [14]:
# --- CONFIGURATION ---
# ⚠️ USER ACTION REQUIRED:
# Upload a PDF to your notebook folder and set the filename below.
# Best results: Use a PRIVATE document (Resume, Internal Memo, Rental Lease).
# If you use a famous book (e.g., Harry Potter), GPT-4 already knows it!
PDF_PATH = "../cirriculum/AI Architect Study Guide - 8 Week Curriculum.pdf" 

if not os.path.exists(PDF_PATH):
    print(f"❌ ERROR: File '{PDF_PATH}' not found. Please upload a PDF and update the PDF_PATH variable.")
    # In a real notebook, we would stop execution here.
    # sys.exit() 
else:
    print(f"✅ Found file: {PDF_PATH}")

✅ Found file: ../cirriculum/AI Architect Study Guide - 8 Week Curriculum.pdf


=============================================================================
##### PHASE 1: EXTRACT (Loading the Raw Data)
=============================================================================

In [15]:
loader = PyMuPDFLoader(PDF_PATH)
raw_docs = loader.load()

print(f"📄 Loaded {len(raw_docs)} pages from {PDF_PATH}.")

📄 Loaded 59 pages from ../cirriculum/AI Architect Study Guide - 8 Week Curriculum.pdf.


=============================================================================
#### PHASE 2: TRANSFORM (Chunking Strategy)
=============================================================================
<br> **ARCHITECT'S NOTE:**
* This is the most critical part of the pipeline.
* Chunk Size 500: Small enough to catch specific details (dates, prices).
* Overlap 50: Prevents sentences from being cut in half at the boundary.
* If this "Transformation" is bad, your "Retrieval" will fail.

In [16]:
print("\n--- [T]RANSFORM: CHUNKING ---")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    add_start_index=True # Keeps metadata about where the chunk came from
)

chunks = text_splitter.split_documents(raw_docs)

print(f"🧩 Split {len(raw_docs)} pages into {len(chunks)} vector-ready chunks.")
print(f"🔍 Inspecting Chunk #0:\n---\n{chunks[0].page_content[:200]}...\n---")


--- [T]RANSFORM: CHUNKING ---
🧩 Split 59 pages into 165 vector-ready chunks.
🔍 Inspecting Chunk #0:
---
The AI Architect Study Guide
8-Week Curriculum: From Data Architect to Head of AI Platform
Hardware Setup: M4 MacBook Air (24GB RAM) - Your unified control and compute plane
Time Investment: ~8-10 hou...
---


=============================================================================
#### PHASE 3: LOAD (Embedding & Vector Storage)
=============================================================================
<br> **ARCHITECT'S NOTE:**
* We are using ChromaDB (an in-memory vector store) for development.
* In Production, you would swap this for:
* pgvector (If you are already on Postgres)
* Pinecone/Weaviate (If you need massive scale or managed services)

In [17]:
print("\n--- [L]OAD: EMBEDDING & INDEXING ---")

# We use the 'small' model to save cost/latency.
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="rag_etl_v1"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3}) # Retrieve top 3 chunks

print("✅ Data successfully loaded into Vector Store.")


--- [L]OAD: EMBEDDING & INDEXING ---
✅ Data successfully loaded into Vector Store.


=============================================================================
##### THE "GLASS BOX" INSPECTOR
=============================================================================
<br>Most tutorials skip this. As an Architect, you must verify what you retrieve.

In [18]:
def inspect_retrieval(query):
    print(f"\n🔎 DEBUG: Executing Retrieval for '{query}'")
    # similarity_search_with_score returns (Document, Score)
    # Score in Chroma: Lower is "Closer" (Better). 0 = Exact Match.
    results = vectorstore.similarity_search_with_score(query, k=3)
    
    for i, (doc, score) in enumerate(results):
        print(f"   [Chunk {i+1} | Dist: {score:.4f}] {doc.page_content[:80]}...")
    
    return results

=============================================================================
##### THE APPLICATION LAYER (The RAG Chain)
=============================================================================
<br> Now that the ETL is done, we build the application logic.

In [19]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# ⚠️ GUARDRAIL PROMPT:
# We explicitly tell the model: "If it's not in the context, say I don't know."
# This prevents it from falling back on its pre-trained knowledge (Hallucination).
template = """
You are a strict QA Assistant. 
Answer the question based ONLY on the following context. 
If the answer is not in the context, say "I don't know based on the provided document."

Context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

=============================================================================
##### INTERACTIVE QA & DIAGNOSTICS
Hint: You can ask questions like the following off the RAG system:
- What are the key takeaways from week 3? Summarize in 3 bullets or less
- What does Phase 2 of the cirriculum cover?
- **Bonus:** What is "Carpe diem"? (this is irrelevant to the document topics, does the agent make something up?)
- **Outlier:** Summarize the entire document in 1 paragraph (Context loss)

=============================================================================

In [22]:
print("\n🤖 RAG SYSTEM READY.")
print("Type 'exit' to quit. Type 'debug' to toggle verbose retrieval.")

while True:
    user_q = input("\nAsk your PDF: ")
    if user_q.lower() == "exit":
        break
    
    # 1. Run Inspection (The "Glass Box")
    # This shows the user EXACTLY what data the LLM is "reading" before it answers.
    inspect_retrieval(user_q)
    
    # 2. Run Generation
    response = rag_chain.invoke(user_q)
    print(f"💡 Answer: {response}")


🤖 RAG SYSTEM READY.
Type 'exit' to quit. Type 'debug' to toggle verbose retrieval.

🔎 DEBUG: Executing Retrieval for 'Summarize the entire document in 1 paragraph'
   [Chunk 1 | Dist: 1.3054] Learn chunking strategies for document processing
Implement basic RAG (Retrieval...
   [Chunk 2 | Dist: 1.3397] # If satisfied: proceed to summarize
       # If not: provide feedback for query...
   [Chunk 3 | Dist: 1.3487] thank you.
End of Study Guide
Download this file as: ai-architect-study-guide.md...
💡 Answer: The document provides a study guide focused on chunking strategies for document processing and the implementation of basic Retrieval Augmented Generation (RAG). It emphasizes the importance of choosing appropriate vector databases for various use cases and introduces core concepts related to embeddings, explaining how text is converted into numerical arrays that maintain semantic meaning through cosine similarity. The guide also includes a Python function for rewriting search queries 